<a href="https://colab.research.google.com/github/av-jones/DimABSA/blob/main/TEST_AJ_PhD_Week3_MultiTask_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhD Week 3: Multi-Task Learning (VA + RDoC) — TEST SET EVALUATION

**Approach:** Predict both VA scores AND RDoC features using shared RoBERTa encoder

**Changes from dev version:**
- Trains on **full training set** (`train_alltasks`)
- Evaluates on **official test set** with gold labels
- No validation split — all data used for training

**Target:** Test RMSE < 0.9


## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate
from google.colab import drive
drive.mount('/content/drive')
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from datasets import Dataset
from torch import nn
from tqdm import tqdm
print('✓ Libraries installed')


## 2. Load Data

In [ ]:
def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

# ── Full training data ────────────────────────────────────────────────────────
train_rest   = load_jsonl('/content/sample_data/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl('/content/sample_data/eng_laptop_train_alltasks.jsonl')

# ── Test set: task1 (aspects) + gold labels ───────────────────────────────────
test_rest    = load_jsonl('/content/sample_data/eng_restaurant_test_task1.jsonl')
test_laptop  = load_jsonl('/content/sample_data/eng_laptop_test_task1.jsonl')
gold_rest    = load_jsonl('/content/sample_data/eng_restaurant_test_gold.jsonl')
gold_laptop  = load_jsonl('/content/sample_data/eng_laptop_test_gold.jsonl')

print(f'Train rest:   {len(train_rest)} sentences')
print(f'Train laptop: {len(train_laptop)} sentences')
print(f'Test rest:    {len(test_rest)} sentences')
print(f'Test laptop:  {len(test_laptop)} sentences')


## 3. Extract Pairs with RDoC Labels

In [ ]:
def compute_rdoc(text):
    t = text.lower()
    pos_words = ['love', 'excellent', 'amazing', 'wonderful', 'great', 'best', 'delicious', 'fantastic', 'perfect']
    neg_words = ['terrible', 'horrible', 'awful', 'worst', 'disappointed', 'poor', 'bad', 'disgusting']
    return (sum(1 for w in pos_words if w in t),
            sum(1 for w in neg_words if w in t))

def extract_multitask(data):
    """For Quadruplet-format files (train_alltasks)"""
    pairs = []
    for item in data:
        rdoc_pos, rdoc_neg = compute_rdoc(item['Text'])
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            v, a = map(float, quad['VA'].split('#'))
            pairs.append({
                'text':     f"{item['Text']} [SEP] {aspect}",
                'valence':  v,
                'arousal':  a,
                'rdoc_pos': float(rdoc_pos),
                'rdoc_neg': float(rdoc_neg)
            })
    return pairs

def extract_multitask_from_gold(data):
    """For Aspect_VA-format files (test gold)"""
    pairs = []
    for item in data:
        rdoc_pos, rdoc_neg = compute_rdoc(item['Text'])
        for av in item['Aspect_VA']:
            v, a = map(float, av['VA'].split('#'))
            pairs.append({
                'text':     f"{item['Text']} [SEP] {av['Aspect']}",
                'valence':  v,
                'arousal':  a,
                'rdoc_pos': float(rdoc_pos),
                'rdoc_neg': float(rdoc_neg)
            })
    return pairs

train_pairs       = extract_multitask(train_rest) + extract_multitask(train_laptop)
test_rest_pairs   = extract_multitask_from_gold(gold_rest)
test_laptop_pairs = extract_multitask_from_gold(gold_laptop)

print(f'Train pairs:       {len(train_pairs)}')
print(f'Test rest pairs:   {len(test_rest_pairs)}')
print(f'Test laptop pairs: {len(test_laptop_pairs)}')
print(f'\nSample multi-task data:')
print(f"  text:    {train_pairs[0]['text'][:80]}")
print(f"  V/A:     {train_pairs[0]['valence']:.2f}, {train_pairs[0]['arousal']:.2f}")
print(f"  RDoC:    pos={train_pairs[0]['rdoc_pos']}, neg={train_pairs[0]['rdoc_neg']}")


## 4. Create Dataset

In [ ]:
train_dataset = Dataset.from_dict({
    'text':     [p['text']     for p in train_pairs],
    'valence':  [p['valence']  for p in train_pairs],
    'arousal':  [p['arousal']  for p in train_pairs],
    'rdoc_pos': [p['rdoc_pos'] for p in train_pairs],
    'rdoc_neg': [p['rdoc_neg'] for p in train_pairs]
})

# No validation dataset — train on full data, evaluate on test
print(f'Train dataset: {len(train_dataset)} samples')


## 5. Multi-Task RoBERTa Model

In [ ]:
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskRoBERTa(nn.Module):
    '''RoBERTa with multi-task heads for V/A + RDoC'''
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        # Shared representation
        self.shared = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 256),
            nn.ReLU()
        )

        # Task 1: V/A prediction (primary)
        self.va_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(256, 2)  # [valence, arousal]
        )

        # Task 2: RDoC prediction (auxiliary)
        self.rdoc_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(256, 2)  # [pos_count, neg_count]
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # [CLS]
        shared_repr = self.shared(pooled)

        va_output = self.va_head(shared_repr)      # Primary task
        rdoc_output = self.rdoc_head(shared_repr)  # Auxiliary task

        return va_output, rdoc_output

model = MultiTaskRoBERTa(MODEL_NAME)
print('✓ Multi-task RoBERTa created')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Tokenize

In [ ]:
def tokenize(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

print('Tokenizing training dataset...')
train_tok = train_dataset.map(tokenize, batched=True)
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'valence', 'arousal', 'rdoc_pos', 'rdoc_neg'])
print('✓ Tokenization complete')


## 7. Train

In [ ]:
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        va_labels   = torch.stack([inputs.pop('valence').float(), inputs.pop('arousal').float()], dim=1)
        rdoc_labels = torch.stack([inputs.pop('rdoc_pos').float(), inputs.pop('rdoc_neg').float()], dim=1)
        va_output, rdoc_output = model(**inputs)
        va_loss   = nn.MSELoss()(va_output,   va_labels)
        rdoc_loss = nn.MSELoss()(rdoc_output, rdoc_labels)
        total_loss = va_loss + 0.3 * rdoc_loss
        return (total_loss, (va_output, rdoc_output)) if return_outputs else total_loss

args = TrainingArguments(
    output_dir='./roberta_multitask',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='no',       # No val set — train on everything
    save_strategy='no',
    report_to='none',
    logging_steps=100,
    label_names=['valence', 'arousal', 'rdoc_pos', 'rdoc_neg']
)

trainer = MultiTaskTrainer(model=model, args=args, train_dataset=train_tok)
print('Starting multi-task training on full dataset (~25 min)...')
print('  Primary task:   V/A prediction (weight=1.0)')
print('  Auxiliary task: RDoC prediction (weight=0.3)')
trainer.train()
print('\n✓ Training complete!')


## 8. Evaluate on Test Set

In [ ]:
def predict_va(text, aspect):
    inputs = tokenizer(f'{text} [SEP] {aspect}', return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(model.roberta.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        va_output, _ = model(**inputs)
    v, a = va_output[0].cpu().numpy()
    return max(1.0, min(9.0, float(v))), max(1.0, min(9.0, float(a)))

def compute_rmse_pairs(pairs, label):
    errors = []
    for p in tqdm(pairs, desc=f'  RMSE {label}', leave=False):
        parts  = p['text'].split(' [SEP] ')
        text   = parts[0]
        aspect = parts[1] if len(parts) > 1 else ''
        pred_v, pred_a = predict_va(text, aspect)
        errors.append((pred_v - p['valence'])**2 + (pred_a - p['arousal'])**2)
    return round(float(np.sqrt(np.mean(errors))), 4) if errors else None

print('Evaluating on TEST set...')
rmse_rest_test   = compute_rmse_pairs(test_rest_pairs,   'Restaurant TEST')
rmse_laptop_test = compute_rmse_pairs(test_laptop_pairs, 'Laptop TEST')

n_rest   = len(test_rest_pairs)
n_laptop = len(test_laptop_pairs)
rmse_overall_test = round(float(np.sqrt(
    (n_rest * rmse_rest_test**2 + n_laptop * rmse_laptop_test**2) / (n_rest + n_laptop)
)), 4)

print(f'\n' + '='*50)
print(f'TEST RMSE  Restaurant : {rmse_rest_test}')
print(f'TEST RMSE  Laptop     : {rmse_laptop_test}')
print(f'TEST RMSE  Overall    : {rmse_overall_test}')
print(f'Target: < 0.9')
print('='*50)


## 9. Save & Download Predictions

In [ ]:
def predict_test_set(test_data, label):
    preds = []
    for item in tqdm(test_data, desc=label):
        av_list = []
        for aspect in item['Aspect']:
            v, a = predict_va(item['Text'], aspect)
            av_list.append({'Aspect': aspect, 'VA': f'{v:.2f}#{a:.2f}'})
        preds.append({'ID': item['ID'], 'Aspect_VA': av_list})
    return preds

def save_predictions(preds, path):
    with open(path, 'w') as f:
        for p in preds:
            f.write(json.dumps(p) + '\n')
    total_aspects = sum(len(p['Aspect_VA']) for p in preds)
    print(f'✓ Saved {len(preds)} sentences ({total_aspects} aspects) to {path}')

print('Generating test predictions...')
rest_preds   = predict_test_set(test_rest,   'Restaurant')
laptop_preds = predict_test_set(test_laptop, 'Laptop')

save_predictions(rest_preds,   '/content/sample_data/pred_eng_restaurant_w3_TEST.jsonl')
save_predictions(laptop_preds, '/content/sample_data/pred_eng_laptop_w3_TEST.jsonl')

from google.colab import files
files.download('/content/sample_data/pred_eng_restaurant_w3_TEST.jsonl')
files.download('/content/sample_data/pred_eng_laptop_w3_TEST.jsonl')
print('\n✓ Files downloaded!')


## 10. Experiment Logger

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📋 EXPERIMENT LOGGER — PhD Week 3: Multi-Task RoBERTa — TEST SET
#
#   ✅ Saves predictions to a timestamped folder in Google Drive
#   ✅ Logs TEST RMSE (restaurant, laptop, overall)
#   ✅ Pulls train loss per epoch from trainer.state
#   ✅ Logs total training runtime
#   ✅ Writes Drive file paths into the Linked Artifact column
#   ✅ Logs everything to your Master Excel sheet
#
# ✏️  Only edit the CONFIG block — nothing else needs changing run to run.
# ═══════════════════════════════════════════════════════════════════════════════

import openpyxl, datetime, os, json, numpy as np
from tqdm import tqdm
from openpyxl.styles import Font

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CONFIG                                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
EXCEL_PATH  = "/content/drive/MyDrive/ML_Experiments/ML_NLP_Experiment_Log.xlsx"
DRIVE_PREDS = "/content/drive/MyDrive/ML_Experiments/Predictions"
SHEET_NAME  = "Experiment Log"
FIRST_ROW   = 4
EXP_NAME    = "PhD_Wk3_MultiTask_RoBERTa_TEST"
RESEARCHER  = "avjones"
NEXT_STEPS  = "Ensemble with MS Mistral predictions (Week 4)"
# ══════════════════════════════════════════════════════════════════════════════


# ── GUARD ─────────────────────────────────────────────────────────────────────
if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(
        f"\n❌  Excel log not found at:\n    {EXCEL_PATH}\n"
        "    → Check EXCEL_PATH above and confirm Drive is mounted (Cell 1)."
    )


# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
def load_jsonl_logger(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

def predict_va_logger(text, aspect):
    inputs = tokenizer(f'{text} [SEP] {aspect}', return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(model.roberta.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        va_output, _ = model(**inputs)
    v, a = va_output[0].cpu().numpy()
    return max(1.0, min(9.0, float(v))), max(1.0, min(9.0, float(a)))

def predict_test_logger(test_data, label):
    preds = []
    for item in tqdm(test_data, desc=f"  Predicting {label}"):
        av_list = []
        for aspect in item['Aspect']:
            v, a = predict_va_logger(item['Text'], aspect)
            av_list.append({'Aspect': aspect, 'VA': f'{v:.2f}#{a:.2f}'})
        preds.append({'ID': item['ID'], 'Aspect_VA': av_list})
    return preds

def save_jsonl_logger(preds, path):
    with open(path, 'w') as f:
        for p in preds:
            f.write(json.dumps(p) + '\n')

def drive_rel(p):
    return p.replace("/content/drive/", "")


# ── STEP 1: TRAINING RUNTIME ──────────────────────────────────────────────────
try:
    runtime_entry = next(
        (e for e in reversed(trainer.state.log_history) if 'train_runtime' in e), None
    )
    if runtime_entry:
        total_secs    = runtime_entry['train_runtime']
        mins, secs    = int(total_secs // 60), int(total_secs % 60)
        runtime_str   = f"{mins}m {secs}s"
        samples_per_s = round(runtime_entry.get('train_samples_per_second', 0), 2)
    else:
        runtime_str, samples_per_s = "N/A", "?"
except Exception as e:
    runtime_str, samples_per_s = f"Error: {e}", "?"

print(f"⏱️   Training time  : {runtime_str}  ({samples_per_s} samples/sec)")


# ── STEP 2: TRAIN LOSS PER EPOCH (no val since eval_strategy='no') ────────────
try:
    log_history    = trainer.state.log_history
    train_by_epoch = {int(e['epoch']): round(e['loss'], 4)
                      for e in log_history if 'loss' in e and 'eval_loss' not in e}
    train_loss_str   = "  ".join([f"E{ep}:{loss}" for ep, loss in sorted(train_by_epoch.items())])
    val_loss_str     = "N/A — eval_strategy=no"
    final_train_loss = list(train_by_epoch.values())[-1] if train_by_epoch else None
    final_val_loss   = None
    best_ckpt        = "N/A — save_strategy=no"
    print(f"📉  Train loss/epoch: {train_loss_str}")
except Exception as e:
    train_loss_str = f"Error: {e}"
    val_loss_str   = "N/A"
    final_train_loss = final_val_loss = None
    best_ckpt = "N/A"


# ── STEP 3: TEST RMSE (already computed in Cell 8) ────────────────────────────
print(f"\n  ✅  Restaurant  TEST RMSE : {rmse_rest_test}")
print(f"  ✅  Laptop      TEST RMSE : {rmse_laptop_test}")
print(f"  ✅  Overall     TEST RMSE : {rmse_overall_test}")


# ── STEP 4: AUTO-GENERATE EXP ID + RUN # ──────────────────────────────────────
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

max_id = 0
for row in ws.iter_rows(min_row=FIRST_ROW, max_col=1, values_only=True):
    v = row[0]
    if v and isinstance(v, str) and v.startswith("EXP-"):
        try: max_id = max(max_id, int(v.split("-")[1]))
        except ValueError: pass
exp_id = f"EXP-{max_id + 1:03d}"

run_num = 1
for row in ws.iter_rows(min_row=FIRST_ROW, min_col=5, max_col=6, values_only=True):
    if row[0] == EXP_NAME:
        try: run_num = max(run_num, int(row[1] or 0) + 1)
        except (TypeError, ValueError): pass

now       = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d_%H%M")
print(f"\n🔖  Exp ID: {exp_id}  |  Run #{run_num}  |  {now.strftime('%Y-%m-%d %H:%M')}")


# ── STEP 5: SAVE PREDICTIONS TO DRIVE ─────────────────────────────────────────
pred_dir         = os.path.join(DRIVE_PREDS, f"{exp_id}_{EXP_NAME}_{timestamp}")
rest_pred_path   = os.path.join(pred_dir, f"pred_restaurant_{exp_id}.jsonl")
laptop_pred_path = os.path.join(pred_dir, f"pred_laptop_{exp_id}.jsonl")
os.makedirs(pred_dir, exist_ok=True)

print(f"\n⏳  Generating predictions (or reusing if already computed)...")
if 'rest_preds' not in dir() or rest_preds is None:
    rest_preds   = predict_test_logger(test_rest,   "Restaurant")
if 'laptop_preds' not in dir() or laptop_preds is None:
    laptop_preds = predict_test_logger(test_laptop, "Laptop")
save_jsonl_logger(rest_preds,   rest_pred_path)
save_jsonl_logger(laptop_preds, laptop_pred_path)

pred_folder_rel  = drive_rel(pred_dir)
rest_pred_rel    = drive_rel(rest_pred_path)
laptop_pred_rel  = drive_rel(laptop_pred_path)
n_rest_aspects   = sum(len(i['Aspect']) for i in test_rest)
n_laptop_aspects = sum(len(i['Aspect']) for i in test_laptop)

print(f"\n💾  Predictions saved:")
print(f"    📁 {pred_folder_rel}/")
print(f"       ├── pred_restaurant_{exp_id}.jsonl  ({len(rest_preds)} sentences, {n_rest_aspects} aspects)")
print(f"       └── pred_laptop_{exp_id}.jsonl      ({len(laptop_preds)} sentences, {n_laptop_aspects} aspects)")


# ── STEP 6: BUILD COLUMN STRINGS ──────────────────────────────────────────────
dataset_size_str = (
    f"Train={len(train_pairs)} | "
    f"Test_Rest={n_rest_aspects} aspects ({len(test_rest)} sentences) | "
    f"Test_Laptop={n_laptop_aspects} aspects ({len(test_laptop)} sentences) | "
    f"No val split — full train_alltasks"
)

custom_metric_str = (
    f"Rest_TEST={rmse_rest_test} | "
    f"Laptop_TEST={rmse_laptop_test} | "
    f"Overall_TEST={rmse_overall_test}"
)

summary_str = (
    f"Runtime: {runtime_str} ({samples_per_s} samp/s) | "
    f"Train loss: {train_loss_str} | "
    f"Val loss: {val_loss_str} | "
    f"BestCkpt: {best_ckpt}"
)

other_hp_str = (
    f"shared=Linear(768→256)+ReLU | va_head=Linear(256→2) | rdoc_head=Linear(256→2) | "
    f"dropout=0.1x3 | loss=MSE(VA) + 0.3*MSE(RDoC) | "
    f"eval_strategy=no | save_strategy=no | max_length=128"
)

artifact_str = (
    f"Folder: {pred_folder_rel} || "
    f"Restaurant: {rest_pred_rel} || "
    f"Laptop: {laptop_pred_rel}"
)


# ── STEP 7: WRITE ROW TO EXCEL ────────────────────────────────────────────────
new_row = [
    exp_id,
    now.strftime("%Y-%m-%d"),
    now.strftime("%H:%M"),
    RESEARCHER,
    EXP_NAME,
    run_num,
    "VA Regression (Multi-Task ABSA)",
    "SemEval Restaurant + Laptop",
    dataset_size_str,
    "PyTorch / HF Transformers",
    MODEL_NAME,
    "Yes",
    args.learning_rate,
    args.per_device_train_batch_size,
    int(args.num_train_epochs),
    "AdamW",
    "LinearWarmup",
    0.1,
    128,
    other_hp_str,
    None,                                # F1
    None,                                # Precision
    None,                                # Recall
    None,                                # Accuracy
    rmse_overall_test,                   # RMSE — test set
    None,                                # BLEU / ROUGE
    custom_metric_str,
    "✅ Success",
    summary_str,
    "None",
    NEXT_STEPS,
    artifact_str,
]

next_row = FIRST_ROW
for row in ws.iter_rows(min_row=FIRST_ROW, max_col=1, values_only=True):
    if row[0]: next_row += 1
    else: break

for col_idx, val in enumerate(new_row, 1):
    ws.cell(row=next_row, column=col_idx, value=val)

artifact_cell           = ws.cell(row=next_row, column=32)
artifact_cell.hyperlink = f"https://drive.google.com/drive/search?q={exp_id}"
artifact_cell.font      = Font(name="Arial", size=10, color="7C3AED", underline="single")

wb.save(EXCEL_PATH)


# ── STEP 8: FINAL SUMMARY ─────────────────────────────────────────────────────
print(f"\n{'═'*64}")
print(f"  ✅  LOGGED SUCCESSFULLY TO EXCEL")
print(f"{'═'*64}")
print(f"  Exp ID           : {exp_id}")
print(f"  Project          : {EXP_NAME}")
print(f"  Run #            : {run_num}")
print(f"  Training time    : {runtime_str}")
print(f"  Final train loss : {final_train_loss}")
print(f"  Final val loss   : {final_val_loss}")
print(f"  Best checkpoint  : {best_ckpt}")
print(f"  RMSE Overall TEST: {rmse_overall_test}")
print(f"  RMSE Rest    TEST: {rmse_rest_test}")
print(f"  RMSE Laptop  TEST: {rmse_laptop_test}")
print(f"  Predictions at   : {pred_folder_rel}")
print(f"  Excel row        : {next_row}  ({EXCEL_PATH})")
print(f"{'═'*64}")


## Summary

**Week 3 Complete — Multi-Task RoBERTa (Test Set):**

1. Trained on full `train_alltasks` with multi-task learning
2. Primary task: V/A prediction | Auxiliary task: RDoC (weight=0.3)
3. Evaluated on official test set with gold labels
4. Predictions saved to Drive

**Next:** Week 4 — Ensemble with MS Mistral predictions
